# NB08 작업증명 — 난이도, 채굴, 가장 긴 체인

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB08-1: 손 채굴기 — 앞 k 비트가 0인 해시를 찾기까지 논스를 몇 번 바꾸는가 (공부용, 난이도는 장난감 수준)
import hashlib

def mine(header_prefix: bytes, bits: int):
    """논스를 0부터 올리며 SHA256(SHA256(prefix||nonce))의 앞 bits 비트가 0인 논스를 찾는다. (논스, 시도 횟수, 해시) 반환."""
    target = 1 << (256 - bits)                 # 「앞 bits 비트가 0」 ⇔ 해시 < 2^(256-bits)
    nonce = 0
    while True:
        h = hashlib.sha256(hashlib.sha256(header_prefix + nonce.to_bytes(4, "little")).digest()).digest()
        if int.from_bytes(h, "big") < target:
            return nonce, nonce + 1, h
        nonce += 1

prefix = b"block#1|prev=00..|merkle=ab..|time=1231006505|"
print(f"{'앞 0비트 k':>8} {'찾은 논스':>10} {'시도 횟수':>10} {'기대 2^k':>10} {'해시 앞 8자':>10}")
for bits in [4, 8, 12, 16, 20]:
    nonce, tries, h = mine(prefix, bits)
    print(f"{bits:>8} {nonce:>10,} {tries:>10,} {2**bits:>10,} {h.hex()[:8]:>10}")

 앞 0비트 k      찾은 논스      시도 횟수     기대 2^k    해시 앞 8자
       4          3          4         16   04468f48
       8        329        330        256   004737bc
      12      6,665      6,666      4,096   000c395c
      16     24,541     24,542     65,536   00002bbf
      20    978,082    978,083  1,048,576   00000fc3


In [ ]:
# NB08-2: 같은 난이도(k=12)로 200개 블록을 캐면 — 시도 횟수의 분포(기하분포)
import math
bits = 12
tries_list = []
for i in range(200):
    _, tries, _ = mine(f"block#{i}|".encode() + prefix, bits)
    tries_list.append(tries)
tries_list.sort()
n = len(tries_list)
mean = sum(tries_list) / n
print(f"k={bits}: 기대 시도 횟수 2^k = {2**bits:,}")
print(f"200회 — 평균 {mean:,.0f}, 중앙값 {tries_list[n//2]:,}, 최소 {tries_list[0]:,}, 최대 {tries_list[-1]:,}")
print(f"기하분포 이론: 평균 {2**bits:,}, 중앙값 ≈ ln2·2^k = {math.log(2)*2**bits:,.0f}, 표준편차 ≈ 2^k = {2**bits:,}")
quick = sum(1 for t in tries_list if t <= 2**bits * 0.1)
slow = sum(1 for t in tries_list if t >= 2**bits * 3)
print(f"기대값의 1/10 안에 끝난 블록: {quick}개 ({quick/2:.0f}%) | 기대값의 3배 이상 걸린 블록: {slow}개 ({slow/2:.0f}%)")

k=12: 기대 시도 횟수 2^k = 4,096
200회 — 평균 4,061, 중앙값 3,058, 최소 27, 최대 25,448
기하분포 이론: 평균 4,096, 중앙값 ≈ ln2·2^k = 2,839, 표준편차 ≈ 2^k = 4,096
기대값의 1/10 안에 끝난 블록: 23개 (12%) | 기대값의 3배 이상 걸린 블록: 9개 (4%)


In [ ]:
# NB08-3: nBits → 목표값 → 난이도 → 해시율 추정. 제네시스(0x1d00ffff)와 2026-10-25 체인 끝(0x17020093)
import math
def nbits_to_target(nbits: int) -> int:
    exponent, coefficient = nbits >> 24, nbits & 0x007FFFFF
    return coefficient * 256 ** (exponent - 3)

T_MAX = nbits_to_target(0x1d00ffff)              # 가장 쉬운 목표값(제네시스의 nBits; powLimit 을 compact 로 줄인 값과 같다)
print("T_max = 0x%064x" % T_MAX)
print("난이도 1에서 블록당 기대 해시 수 2^256/T_max = %.6f × 2^32" % (2**256 / T_MAX / 2**32))

for name, nbits in [("제네시스(2009-01-03)", 0x1d00ffff), ("높이 972,958 (2026-10-25 기준)", 0x17020093)]:
    T = nbits_to_target(nbits)
    D = T_MAX / T
    p = (T + 1) / 2**256
    print(f"\n{name}: nBits = 0x{nbits:08x}")
    print("  목표값 T = 0x%064x" % T)
    print("  해시 한 번의 성공 확률 p = %.3e  (앞 0비트 ≈ %.1f개)" % (p, 256 - math.log2(T + 1)))
    print("  난이도 D = T_max/T = %.1f" % D)
    print("  블록당 기대 해시 수 ≈ D·2^32 = %.3e" % (D * 2**32))
    H = D * 2**32 / 600
    print("  600초 가정 해시율 추정 D·2^32/600 = %.4e H/s" % H + (" = %.0f EH/s" % (H / 1e18) if H >= 1e15 else " = %.1f MH/s" % (H / 1e6)))

T_max = 0x00000000ffff0000000000000000000000000000000000000000000000000000
난이도 1에서 블록당 기대 해시 수 2^256/T_max = 1.000015 × 2^32

제네시스(2009-01-03): nBits = 0x1d00ffff
  목표값 T = 0x00000000ffff0000000000000000000000000000000000000000000000000000
  해시 한 번의 성공 확률 p = 2.328e-10  (앞 0비트 ≈ 32.0개)
  난이도 D = T_max/T = 1.0
  블록당 기대 해시 수 ≈ D·2^32 = 4.295e+09
  600초 가정 해시율 추정 D·2^32/600 = 7.1583e+06 H/s = 7.2 MH/s

높이 972,958 (2026-10-25 기준): nBits = 0x17020093
  목표값 T = 0x0000000000000000000200930000000000000000000000000000000000000000
  해시 한 번의 성공 확률 p = 1.656e-24  (앞 0비트 ≈ 79.0개)
  난이도 D = T_max/T = 140577680051919.6
  블록당 기대 해시 수 ≈ D·2^32 = 6.038e+23
  600초 가정 해시율 추정 D·2^32/600 = 1.0063e+21 H/s = 1006 EH/s


In [ ]:
# NB08-4: 조정 규칙 시뮬레이션 — 해시율이 갑자기 16배가 되면 난이도는 몇 번의 조정으로 따라가나 (Core 규칙 그대로)
TIMESPAN, SPACING, INTERVAL = 1_209_600, 600, 2016      # chainparams.cpp·params.h 의 값
def retarget(T_old: int, actual_timespan: int) -> int:
    """pow.cpp CalculateNextWorkRequired: 소요 시간을 [timespan/4, timespan*4]로 자르고 비례 조정, powLimit 상한."""
    actual = max(TIMESPAN // 4, min(TIMESPAN * 4, actual_timespan))
    T_new = T_old * actual // TIMESPAN
    return min(T_new, T_MAX)

T = T_MAX // 1000                      # 어떤 출발 난이도(1000)에서
H1 = 2**256 / (T + 1) / 600            # 그 난이도에서 정확히 600초/블록이 나오는 해시율(해시/초)을 기준 1로 둔다
hashrate = 1.0
print(f"{'조정 회차':>6} {'난이도':>10} {'구간 평균 블록 간격(초)':>22} {'다음 난이도':>12}")
for period in range(1, 7):
    if period == 2:
        hashrate = 16.0                # 2회차 구간부터 해시율이 16배로 뛴다
    D = T_MAX / T
    expected_hashes_per_block = 2**256 / (T + 1)
    seconds_per_block = expected_hashes_per_block / (hashrate * H1)
    actual_timespan = int(seconds_per_block * INTERVAL)
    T_next = retarget(T, actual_timespan)
    print(f"{period:>6} {D:>10,.0f} {seconds_per_block:>22,.0f} {T_MAX / T_next:>12,.0f}")
    T = T_next

 조정 회차        난이도         구간 평균 블록 간격(초)       다음 난이도
     1      1,000                    600        1,000
     2      1,000                     38        4,000
     3      4,000                    150       16,000
     4     16,000                    600       16,000
     5     16,000                    600       16,000
     6     16,000                    600       16,000
